# Family B screener — the strategy as a daily instrument

`famb_strategy_showcase` presents the two surviving structures as backtests.
This notebook is the other half: what they say to do on a **given day**,
including today. Same engine (`famb_common`), same signal, one date.

```
conda run -n stir python notebooks/backtests/famb_screener.py --date 2026-07-28
conda run -n stir python notebooks/backtests/famb_screener.py --date live
conda run -n stir python notebooks/backtests/famb_screener.py --date live --json
```

Two sleeves. The **carry short** (FLY25 on the front quarterly) is always on,
so the screener reports its position and its roll clock rather than a signal.
The **richness fade** (STRG75 on the front quarterly) has an entry rule, and
watching it is what this exists for.

In [1]:
import sys

import numpy as np
import pandas as pd

sys.path.append("../../")
sys.path.append(".")
import famb_screener as scr                                  # noqa: E402

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

CONFIG = dict(
    date="2023-03-20",      # a date, or "live"
    history=250,            # sessions of context for the standing level
    lots=100,
    thr_bp=scr.PREREG["thr_bp"],
)
CONFIG

{'date': '2023-03-20', 'history': 250, 'lots': 100, 'thr_bp': 4.0}

## 1. A day the strategy actually traded

2023-03-20 is the post-SVB richness spike the findings doc records as the
fade's best entry (+44bp collected as the panic premium converged). If the
screener is wired to the same signal, it has to see it.

In [2]:
ctx = scr.load_context(CONFIG["date"], history=CONFIG["history"])
ideas = scr.screen(ctx, thr_bp=CONFIG["thr_bp"])
carry = scr.carry_state(ctx)
print(scr.format_report(ctx, ideas, carry, lots=CONFIG["lots"]))

FAMILY B SCREENER  |  HISTORICAL as of 2023-03-20
  screen date 2023-03-20   history 2022-03-22 -> 2023-03-20 (250 sessions)

-- SLEEVE 1: carry (always on) -----------------------------------------------------------------
  SHORT FLY25 SFRM23 94.25/94.50/94.75   held since 2023-03-07
  mark 2.00bp   fair 0.00bp   richness +2.00bp
  expiry 2023-06-16   ROLL ON 2023-06-13 (85 calendar days)   roll cost 1.00bp

-- SLEEVE 2: richness fade (signal-gated) -----------------------------------------------------
  cell        sym      dte    rich   level     dev     z  net@1x xcost  risk  state
  ---------------------------------------------------------------------------------
 *STRG75 Q1   SFRM23    88  +28.73   +1.19  +27.54  +6.4  +21.05  43.1   UNB  ACTIONABLE
  FLY50 Q1    SFRM23    88   +9.98   -0.06  +10.04  +2.6   +6.53   7.5   def  ACTIONABLE
  STRG50 Q1   SFRM23    88  +17.43   +1.92  +15.51  +3.1  +11.13  23.3   UNB  ACTIONABLE
  FLY25 Q1    SFRM23    88   +2.00   -0.00   +2.00  +1.4

## 2. What the screen is actually measuring

The column that matters is not `rich` but `dev` — richness minus the cell's
own standing level. Rank-2 and rank-3 contracts carry a large permanent
richness because the off-lattice premium grows with days to expiry (the
feasibility frontier, measured at +1.9 / +9.1 / +19.2bp by rank in the
findings). Threshold-gating the raw number out there is not a convergence
signal, it is a standing short of that premium — the passive tail-short the
referee's ruling killed.

In [3]:
tab = pd.DataFrame([{
    "cell": f"{i.book} Q{i.rank}", "symbol": i.symbol, "dte": i.dte,
    "rich_bp": round(i.rich_bp, 2), "level_bp": round(i.level_bp, 2),
    "dev_bp": round(i.dev_bp, 2),
    "z": round(i.z_dev, 2) if np.isfinite(i.z_dev) else np.nan,
    "defined_risk": i.defined_risk, "state": i.state,
} for i in ideas if i.state != "NO-DATA"])
print(tab.to_string(index=False))
print("\nstanding level by rank (the frontier, in premium bp):")
print(tab.assign(rank=tab["cell"].str[-1])
      .groupby("rank")["level_bp"].agg(["mean", "min", "max"]).round(2)
      .to_string())

     cell symbol  dte  rich_bp  level_bp  dev_bp     z  defined_risk      state
STRG75 Q1 SFRM23   88    28.73      1.19   27.54  6.43         False ACTIONABLE
 FLY50 Q1 SFRM23   88     9.98     -0.06   10.04  2.57          True ACTIONABLE
STRG50 Q1 SFRM23   88    17.43      1.92   15.51  3.14         False ACTIONABLE
 FLY25 Q1 SFRM23   88     2.00     -0.00    2.00  1.36          True      WATCH
  DFLY Q1 SFRM23   88    -2.48     -0.01   -2.47 -1.38          True      WATCH

standing level by rank (the frontier, in premium bp):
      mean   min   max
rank                  
1     0.61 -0.06  1.92


## 3. The machine-readable form

`--json` emits one record per cell, so the screen can drive a sheet, an alert
or an order ticket rather than a human reading a table.

In [4]:
recs = scr.to_records(ideas)
best = next((r for r in recs if r["state"] == "ACTIONABLE"), recs[0])
print(f"{len(recs)} records; top one:")
for k in ("book", "rank", "symbol", "state", "action", "rich_bp", "dev_bp",
          "z_dev", "net_target_bp", "edge_mult", "exit_rich_bp",
          "max_hold_date", "defined_risk", "prereg"):
    v = best[k]
    print(f"  {k:16s} {round(v, 4) if isinstance(v, float) else v}")
print("  legs:")
for leg in best["legs"]:
    print(f"      {leg}")

15 records; top one:
  book             STRG75
  rank             1
  symbol           SFRM23
  state            ACTIONABLE
  action           SELL STRG75
  rich_bp          28.7344
  dev_bp           27.5405
  z_dev            6.4272
  net_target_bp    21.0508
  edge_mult        43.1016
  exit_rich_bp     7.1836
  max_hold_date    2023-04-10
  defined_risk     False
  prereg           True
  legs:
      {'right': 'P', 'strike_price': 93.75, 'weight': 1.0}
      {'right': 'C', 'strike_price': 95.25, 'weight': 1.0}


## 4. A day it says nothing

The screener's normal output is "no trade", and that has to be
distinguishable from "no data" — the first version of this tool printed the
former when it meant the latter. A quiet screen with a named closest cell is
a working screen; a NO DATA banner is a broken pipeline.

In [5]:
ctx_q = scr.load_context("2026-07-28", history=CONFIG["history"])
ideas_q = scr.screen(ctx_q, thr_bp=CONFIG["thr_bp"], ranks=(1,))
print(scr.format_report(ctx_q, ideas_q, scr.carry_state(ctx_q),
                        lots=CONFIG["lots"]))

FAMILY B SCREENER  |  HISTORICAL as of 2026-07-28
  screen date 2026-07-28   history 2025-08-13 -> 2026-07-28 (250 sessions)

-- SLEEVE 1: carry (always on) -----------------------------------------------------------------
  SHORT FLY25 SFRU26 96.00/96.25/96.50   held since 2026-06-09
  mark 8.98bp   fair 7.82bp   richness +1.16bp
  expiry 2026-09-11   ROLL ON 2026-09-08 (42 calendar days)   roll cost 1.00bp

-- SLEEVE 2: richness fade (signal-gated) -----------------------------------------------------
  cell        sym      dte    rich   level     dev     z  net@1x xcost  risk  state
  ---------------------------------------------------------------------------------
 *STRG75 Q1   SFRU26    45   +1.24   +0.75   +0.49  +0.5   +0.43   1.9   UNB  WATCH (needs +2.76bp more rich)
  FLY25 Q1    SFRU26    45   +1.16   -0.08   +1.24  +0.8   -0.07   0.9   def  WATCH (needs +2.76bp more dev)
  FLY50 Q1    SFRU26    45   -0.60   -0.62   +0.02  +0.0   -0.99   0.0   def  WATCH (needs +3.98bp more 

## How to read any of this

The fade's parameters won a 720-config search: DSR 0.000, house verdict
SELECTION-ARTIFACT. Its mechanism is measured — 7–18 session richness
half-lives, frontier-consistent levels by rank, positive skew through SVB —
but the exact numbers are selection-inflated, and the carry short's NW t is
1.51. Running the pre-registered cell unchanged, forward, is the one-trial
test that discharges the penalty; every other cell the screen prints is an
exploratory read with no mandate at all.